# How a GPU Runs Deep Learning: Memory, the Roofline, and a First Kernel

The runnable code of [How a GPU Runs Deep Learning: Memory, the Roofline, and a First Kernel](https://residual-stream.lvtan71.workers.dev/posts/how-a-gpu-runs-deep-learning/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.



**Hardware:** Needs an Apple-silicon Mac: the kernels are written in Metal, which Colab can't run. Run it locally with Jupyter (see the repo's README); the cells that are only arithmetic run anywhere. About 2 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "how-a-gpu-runs-deep-learning"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/how-a-gpu-runs-deep-learning/gpu_util.py']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Threads, warps, blocks, grids

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/how-a-gpu-runs-deep-learning/#threads-warps-blocks-grids)

In [ ]:
%%writefile kernels.metal
#include <metal_stdlib>
using namespace metal;

// Metal launches exactly `threads` threads, so these kernels skip bounds
// checks; CUDA rounds the grid up to whole blocks and needs `if (i < n)`.

// One thread per element: the GPU's "hello world".
kernel void vector_add(device float* out, device const float* a, device const float* b,
                       uint i [[thread_position_in_grid]]) {
    out[i] = a[i] + b[i];
}

// Every thread copies one element, but reads it from position (i * stride) mod n.
// Stride 1: neighbouring threads read neighbouring addresses.
kernel void strided_copy(device float* out, device const float* a, constant uint& n,
                         constant uint& stride, uint i [[thread_position_in_grid]]) {
    out[i] = a[(i * stride) % n];
}

// Matrix multiply, one thread per output element, reading straight from memory.
kernel void matmul_naive(device const float* A, device const float* B, device float* C,
                         constant uint& n, uint2 p [[thread_position_in_grid]]) {
    float acc = 0.0f;
    for (uint k = 0; k < n; k++)
        acc += A[p.y * n + k] * B[k * n + p.x];
    C[p.y * n + p.x] = acc;
}

// The same, but each 16 x 16 group of threads first copies a tile of A and a
// tile of B into fast on-chip threadgroup memory, then reuses each loaded
// number 16 times. Assumes n is a multiple of 16.
kernel void matmul_tiled(device const float* A, device const float* B, device float* C,
                         constant uint& n, uint2 p [[thread_position_in_grid]],
                         uint2 t [[thread_position_in_threadgroup]]) {
    threadgroup float As[16][16];
    threadgroup float Bs[16][16];
    float acc = 0.0f;
    for (uint k0 = 0; k0 < n; k0 += 16) {
        As[t.y][t.x] = A[p.y * n + k0 + t.x];
        Bs[t.y][t.x] = B[(k0 + t.y) * n + p.x];
        threadgroup_barrier(mem_flags::mem_threadgroup);     // wait until the tiles are loaded
        for (uint k = 0; k < 16; k++)
            acc += As[t.y][k] * Bs[k][t.x];
        threadgroup_barrier(mem_flags::mem_threadgroup);     // wait before overwriting them
    }
    C[p.y * n + p.x] = acc;
}

// A bigger tile, 32 x 32: twice the reuse of each number read from memory.
kernel void matmul_tiled32(device const float* A, device const float* B, device float* C,
                         constant uint& n, uint2 p [[thread_position_in_grid]],
                         uint2 t [[thread_position_in_threadgroup]]) {
    threadgroup float As[32][32];
    threadgroup float Bs[32][32];
    float acc = 0.0f;
    for (uint k0 = 0; k0 < n; k0 += 32) {
        As[t.y][t.x] = A[p.y * n + k0 + t.x];
        Bs[t.y][t.x] = B[(k0 + t.y) * n + p.x];
        threadgroup_barrier(mem_flags::mem_threadgroup);     // wait until the tiles are loaded
        for (uint k = 0; k < 32; k++)
            acc += As[t.y][k] * Bs[k][t.x];
        threadgroup_barrier(mem_flags::mem_threadgroup);     // wait before overwriting them
    }
    C[p.y * n + p.x] = acc;
}

// One level further: each thread keeps a 4 x 4 block of outputs in registers,
// so a 16 x 16 group computes a 64 x 64 tile of C. Per step of k, a thread
// loads 4 numbers of A and 4 of B from threadgroup memory and does 16
// multiply-adds with them, instead of 2 loads per multiply-add.
// Assumes n is a multiple of 64.
kernel void matmul_registers(device const float* A, device const float* B, device float* C,
                             constant uint& n, uint2 g [[threadgroup_position_in_grid]],
                             uint2 t [[thread_position_in_threadgroup]]) {
    threadgroup float As[64][16];
    threadgroup float Bs[16][64];
    float acc[4][4] = {};
    uint row0 = g.y * 64, col0 = g.x * 64, id = t.y * 16 + t.x;
    for (uint k0 = 0; k0 < n; k0 += 16) {
        for (uint e = id; e < 1024; e += 256) {              // 256 threads load 1,024 numbers each
            As[e / 16][e % 16] = A[(row0 + e / 16) * n + k0 + e % 16];
            Bs[e / 64][e % 64] = B[(k0 + e / 64) * n + col0 + e % 64];
        }
        threadgroup_barrier(mem_flags::mem_threadgroup);
        for (uint k = 0; k < 16; k++) {
            float a[4], b[4];
            for (uint i = 0; i < 4; i++) a[i] = As[t.y * 4 + i][k];
            for (uint j = 0; j < 4; j++) b[j] = Bs[k][t.x * 4 + j];
            for (uint i = 0; i < 4; i++)
                for (uint j = 0; j < 4; j++)
                    acc[i][j] += a[i] * b[j];
        }
        threadgroup_barrier(mem_flags::mem_threadgroup);
    }
    for (uint i = 0; i < 4; i++)
        for (uint j = 0; j < 4; j++)
            C[(row0 + t.y * 4 + i) * n + col0 + t.x * 4 + j] = acc[i][j];
}

In [ ]:
%%writefile gpu_util.py
import time
import torch

lib = torch.mps.compile_shader(open("kernels.metal").read())

def timed(fn, trials=50):
    """Median seconds per call, waiting for the GPU before each clock reading."""
    fn(); torch.mps.synchronize()
    times = []
    for _ in range(trials):
        t0 = time.perf_counter(); fn(); torch.mps.synchronize()
        times.append(time.perf_counter() - t0)
    return sorted(times)[trials // 2]

def timed_queued(fn, reps=200):
    """Seconds per call when `reps` calls are queued back to back and the GPU
    is waited on once: throughput, as a training loop sees it."""
    fn(); torch.mps.synchronize()
    t0 = time.perf_counter()
    for _ in range(reps):
        fn()
    torch.mps.synchronize()
    return (time.perf_counter() - t0) / reps

## Memory decides: bandwidth and coalescing

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/how-a-gpu-runs-deep-learning/#memory-decides-bandwidth-and-coalescing)

In [ ]:
import torch
from gpu_util import lib, timed

n = 1 << 26                                             # 64M floats, 268 MB per array
a, b = torch.randn(n, device="mps"), torch.randn(n, device="mps")
out = torch.empty_like(a)
t = timed(lambda: lib.vector_add(out, a, b))
print(f"vector_add: read 2 arrays, write 1, in {t * 1e3:.1f} ms = {3 * n * 4 / t / 1e9:.0f} GB/s")
t = timed(lambda: out.copy_(a))
print(f"PyTorch copy: {2 * n * 4 / t / 1e9:.0f} GB/s")
# Odd strides are coprime with n, so every element is still read exactly once.
for stride in (1, 3, 9, 33, 1025):
    t = timed(lambda: lib.strided_copy(out, a, n, stride))
    print(f"strided_copy, stride {stride:4d}: {2 * n * 4 / t / 1e9:5.0f} GB/s")

The post printed:

```text
vector_add: read 2 arrays, write 1, in 8.8 ms = 91 GB/s
PyTorch copy: 92 GB/s
strided_copy, stride    1:    93 GB/s
strided_copy, stride    3:    48 GB/s
strided_copy, stride    9:    17 GB/s
strided_copy, stride   33:    10 GB/s
strided_copy, stride 1025:     9 GB/s
```

## The roofline

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/how-a-gpu-runs-deep-learning/#the-roofline)

In [ ]:
import torch
from gpu_util import timed_queued as timed

# Operations from very little math per byte to a lot, measured on this GPU in
# float32, each queued 200 times and waited on once (throughput). Intensity = FLOPs / bytes that must move between memory and the chip.
rows = []
x = torch.randn(1 << 24, device="mps"); y = torch.randn_like(x)
t = timed(lambda: x + y)
rows.append(("vector add", len(x), 3 * 4 * len(x), t))
for n in (4096, 8192):
    M, v = torch.randn(n, n, device="mps"), torch.randn(n, device="mps")
    t = timed(lambda: M @ v)
    rows.append((f"matrix-vector, n={n}", 2 * n * n, 4 * (n * n + 2 * n), t))
for n in (64, 128, 256, 512, 1024, 2048, 4096):
    A, B = torch.randn(n, n, device="mps"), torch.randn(n, n, device="mps")
    t = timed(lambda: A @ B)
    rows.append((f"matmul, n={n}", 2 * n ** 3, 3 * 4 * n * n, t))
# The smallest matmul again, but 1,024 of them batched into one call.
A, B = torch.randn(1024, 64, 64, device="mps"), torch.randn(1024, 64, 64, device="mps")
t = timed(lambda: torch.bmm(A, B))
rows.append(("1,024 matmuls n=64, bmm", 1024 * 2 * 64 ** 3, 1024 * 3 * 4 * 64 * 64, t))
for name, flops, bytes_, t in rows:
    print(f"{name:22s} intensity {flops / bytes_:8.2f} FLOP/byte   {flops / t / 1e9:7.1f} GFLOP/s   "
          f"{bytes_ / t / 1e9:6.1f} GB/s")

The post printed:

```text
vector add             intensity     0.08 FLOP/byte       6.1 GFLOP/s     73.2 GB/s
matrix-vector, n=4096  intensity     0.50 FLOP/byte      39.6 GFLOP/s     79.2 GB/s
matrix-vector, n=8192  intensity     0.50 FLOP/byte      39.7 GFLOP/s     79.5 GB/s
matmul, n=64           intensity    10.67 FLOP/byte      25.3 GFLOP/s      2.4 GB/s
matmul, n=128          intensity    21.33 FLOP/byte     273.8 GFLOP/s     12.8 GB/s
matmul, n=256          intensity    42.67 FLOP/byte    1325.1 GFLOP/s     31.1 GB/s
matmul, n=512          intensity    85.33 FLOP/byte    2778.1 GFLOP/s     32.6 GB/s
matmul, n=1024         intensity   170.67 FLOP/byte    2749.2 GFLOP/s     16.1 GB/s
matmul, n=2048         intensity   341.33 FLOP/byte    2877.5 GFLOP/s      8.4 GB/s
matmul, n=4096         intensity   682.67 FLOP/byte    2662.8 GFLOP/s      3.9 GB/s
1,024 matmuls n=64, bmm intensity    10.67 FLOP/byte     728.5 GFLOP/s     68.3 GB/s
```

## A matrix multiply, from naive to tiled

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/how-a-gpu-runs-deep-learning/#a-matrix-multiply-from-naive-to-tiled)

In [ ]:
import torch
from gpu_util import lib, timed_queued as timed

for n in (512, 1024, 2048):
    A, B = torch.randn(n, n, device="mps"), torch.randn(n, n, device="mps")
    exact = A.cpu().double() @ B.cpu().double()        # the reference, in float64 on the CPU
    row, worst = [f"n = {n:4d}:"], 0.0
    for name, run in [("naive", lambda C: lib.matmul_naive(A, B, C, n, threads=(n, n), group_size=(16, 16))),
                      ("tiled", lambda C: lib.matmul_tiled(A, B, C, n, threads=(n, n), group_size=(16, 16))),
                      ("tiled 32", lambda C: lib.matmul_tiled32(A, B, C, n, threads=(n, n), group_size=(32, 32))),
                      ("registers", lambda C: lib.matmul_registers(A, B, C, n, threads=(n // 4, n // 4),
                                                                   group_size=(16, 16))),
                      ("PyTorch", lambda C: torch.mm(A, B, out=C))]:
        C = torch.empty(n, n, device="mps")
        t = timed(lambda: run(C))
        worst = max(worst, (C.cpu().double() - exact).abs().max().item() / exact.abs().max().item())
        row.append(f"{name} {2 * n ** 3 / t / 1e9:5.0f} GFLOP/s")
    print("   ".join(row) + f"   (all within {worst:.0e} of float64)")

The post printed:

```text
n =  512:   naive   237 GFLOP/s   tiled   518 GFLOP/s   tiled 32   561 GFLOP/s   registers  1325 GFLOP/s   PyTorch  2869 GFLOP/s   (all within 1e-06 of float64)
n = 1024:   naive   291 GFLOP/s   tiled   514 GFLOP/s   tiled 32   581 GFLOP/s   registers  1308 GFLOP/s   PyTorch  3081 GFLOP/s   (all within 1e-06 of float64)
n = 2048:   naive   269 GFLOP/s   tiled   491 GFLOP/s   tiled 32   517 GFLOP/s   registers  1201 GFLOP/s   PyTorch  2867 GFLOP/s   (all within 2e-06 of float64)
```

In [ ]:
# How tiling changes a matmul's arithmetic intensity, counted as float32
# reads from main memory with no help from caches: an upper bound on traffic.
# Each output element takes n multiply-adds (2n FLOPs) over a row of A and a
# column of B: 2n numbers read, if nothing is reused.
ridge_here, ridge_h100 = 2878 / 93, 989e3 / 3350      # measured laptop GPU; H100 datasheet
print(f"ridge points: this GPU {ridge_here:.0f} FLOP/byte, H100 {ridge_h100:.0f} FLOP/byte\n")
print(f"{'C tile':>9s} {'reads per output':>17s} {'intensity':>13s} {'threads, one output each':>26s}")
for T in (1, 16, 32, 64, 128):
    # A T x T tile of outputs reads a T-row strip of A and a T-column strip of
    # B: 2nT numbers for T*T outputs, so each number read is used T times.
    reads_per_output = 2 / T                          # in units of n numbers
    intensity = 2 / (reads_per_output * 4)            # FLOPs per byte
    label = "naive" if T == 1 else f"{T} x {T}"
    print(f"{label:>9s} {reads_per_output:>12.3f} x n {intensity:>8.2f} FLOP/B {T * T:>18,d}")

The post printed:

```text
ridge points: this GPU 31 FLOP/byte, H100 295 FLOP/byte

   C tile  reads per output     intensity   threads, one output each
    naive        2.000 x n     0.25 FLOP/B                  1
  16 x 16        0.125 x n     4.00 FLOP/B                256
  32 x 32        0.062 x n     8.00 FLOP/B              1,024
  64 x 64        0.031 x n    16.00 FLOP/B              4,096
128 x 128        0.016 x n    32.00 FLOP/B             16,384
```

In [ ]:
import torch

# How much threadgroup (shared) memory will this GPU give one block?
for kb in (16, 32, 64):
    source = f"""
    #include <metal_stdlib>
    using namespace metal;
    kernel void probe(device float* out, uint i [[thread_position_in_grid]],
                      uint t [[thread_position_in_threadgroup]]) {{
        threadgroup float buf[{kb * 256}];                   // kb kilobytes of float32
        buf[t] = i;
        threadgroup_barrier(mem_flags::mem_threadgroup);
        out[i] = buf[(t + 1) % 32] + buf[{kb * 256 - 1}];
    }}"""
    try:
        out = torch.zeros(64, device="mps")
        torch.mps.compile_shader(source).probe(out, threads=64, group_size=32)
        torch.mps.synchronize()
        print(f"{kb} KB of threadgroup memory: ok")
    except Exception as e:
        print(f"{kb} KB of threadgroup memory: refused ({str(e).split('\"')[1][:60]})")

The post printed:

```text
16 KB of threadgroup memory: ok
32 KB of threadgroup memory: ok
64 KB of threadgroup memory: refused (Threadgroup memory size (65536) exceeds the maximum threadgr)
```

## Why matrix shapes matter

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/how-a-gpu-runs-deep-learning/#why-matrix-shapes-matter)

In [ ]:
import math
# Arithmetic for an H100: a matmul kernel splits the output into 128 x 128
# tiles and hands them to 132 SMs in waves. A last, partly empty wave is paid
# for in full.
for n in (1024, 1408, 1536, 4096, 4097, 4224):
    per_side = math.ceil(n / 128)
    tiles = per_side ** 2
    waves = math.ceil(tiles / 132)
    useful = (n / (per_side * 128)) ** 2              # edge tiles are partly padding
    print(f"n = {n}: {tiles:4d} tiles -> {waves:2d} waves; SMs busy {tiles / (waves * 132):4.0%}, "
          f"useful work in the tiles {useful:4.0%}, overall {useful * tiles / (waves * 132):4.0%}")

The post printed:

```text
n = 1024:   64 tiles ->  1 waves; SMs busy  48%, useful work in the tiles 100%, overall  48%
n = 1408:  121 tiles ->  1 waves; SMs busy  92%, useful work in the tiles 100%, overall  92%
n = 1536:  144 tiles ->  2 waves; SMs busy  55%, useful work in the tiles 100%, overall  55%
n = 4096: 1024 tiles ->  8 waves; SMs busy  97%, useful work in the tiles 100%, overall  97%
n = 4097: 1089 tiles ->  9 waves; SMs busy  92%, useful work in the tiles  94%, overall  86%
n = 4224: 1089 tiles ->  9 waves; SMs busy  92%, useful work in the tiles 100%, overall  92%
```